[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/02_tag_1_reinforcement_learning_mini_pong.ipynb)

# Tag 1 Reinforcement Learning Mini Pong

Dieses Notebook zeigt ein kleines Pong-ähnliches Reinforcement-Learning-Beispiel ohne Gym, Atari oder GPU. Der Ball bewegt sich zwischen linker und rechter Seite. Der Agent steuert das rechte Paddle und lernt, den Ball zurückzuspielen.

## 1. Grundbegriffe

- **Agent**: entscheidet, ob das rechte Paddle hoch, runter oder stehen bleibt.
- **Environment**: die Pong-Welt mit Ball, linkem Paddle, rechtem Paddle und Wänden.
- **State**: Position/Richtung des Balls und Position des Agent-Paddles.
- **Action**: `hoch`, `stehen` oder `runter`.
- **Reward**: `+1` bei Treffer rechts, `-1` bei Verpassen, kleine Schrittstrafe für langes Zögern.
- **Policy**: Strategie zur Wahl der nächsten Aktion.
- **Episode**: ein Ballwechsel bis Miss oder maximale Schrittzahl.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
import ipywidgets as widgets

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (7, 4)
ACTIONS = [-1, 0, 1]  # hoch, stehen, runter
ACTION_NAMES = {-1: 'hoch', 0: 'stehen', 1: 'runter'}
print('Setup abgeschlossen.')

## 2. Pong-Umgebung

Die Umgebung ist bewusst klein, funktioniert aber näher am Spiel: Der Ball prallt oben/unten ab, ein einfaches linkes Paddle spielt automatisch zurück, und der Agent muss rechts den Ball treffen. Dadurch sieht man deutlich, ob der Agent wirklich etwas lernt.

In [ ]:
class MiniPongEnv:
    def __init__(self, width=12, height=8, max_steps=250, step_penalty=-0.002, seed=RANDOM_STATE):
        self.width = int(width)
        self.height = int(height)
        self.max_steps = int(max_steps)
        self.step_penalty = float(step_penalty)
        self.rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        self.ball_x = self.width // 2
        self.ball_y = int(self.rng.integers(1, self.height - 1))
        self.vx = int(self.rng.choice([-1, 1]))
        self.vy = int(self.rng.choice([-1, 1]))
        self.left_paddle_y = self.height // 2
        self.right_paddle_y = self.height // 2
        self.steps = 0
        self.agent_hits = 0
        self.left_hits = 0
        return self.state()

    def state(self):
        # Kleine Q-Tabelle: Ballposition, Ballrichtung und Position des rechten Paddles.
        return (self.ball_x, self.ball_y, self.vx, self.vy, self.right_paddle_y)

    def _move_auto_left_paddle(self):
        # Einfacher Gegner: folgt dem Ball langsam. Dadurch bleibt der Fokus auf dem Agenten rechts.
        if self.ball_y < self.left_paddle_y:
            self.left_paddle_y -= 1
        elif self.ball_y > self.left_paddle_y:
            self.left_paddle_y += 1
        self.left_paddle_y = int(np.clip(self.left_paddle_y, 1, self.height - 2))

    def step(self, action):
        action = int(action)
        self.right_paddle_y = int(np.clip(self.right_paddle_y + action, 1, self.height - 2))
        self._move_auto_left_paddle()

        self.ball_x += self.vx
        self.ball_y += self.vy
        self.steps += 1
        reward = self.step_penalty
        done = False

        # Obere und untere Wand.
        if self.ball_y <= 0 or self.ball_y >= self.height - 1:
            self.vy *= -1
            self.ball_y = int(np.clip(self.ball_y, 0, self.height - 1))

        # Linkes Paddle spielt automatisch zurück.
        if self.ball_x <= 1:
            if abs(self.ball_y - self.left_paddle_y) <= 1:
                self.vx = 1
                self.ball_x = 1
                self.left_hits += 1
            else:
                # Linker Miss ist für den Agenten kein Fehler, startet aber einen neuen Ballwechsel.
                reward += 0.2
                self.vx = 1
                self.ball_x = 1
                self.ball_y = int(self.rng.integers(1, self.height - 1))

        # Rechtes Paddle wird vom Agenten gesteuert.
        if self.ball_x >= self.width - 2:
            if abs(self.ball_y - self.right_paddle_y) <= 1:
                reward = 1.0
                self.vx = -1
                self.ball_x = self.width - 2
                self.agent_hits += 1
            else:
                reward = -1.0
                done = True

        if self.steps >= self.max_steps:
            done = True
        return self.state(), reward, done, {'agent_hits': self.agent_hits, 'left_hits': self.left_hits}

    def render_array(self):
        grid = np.zeros((self.height, self.width))
        # Paddles
        for dy in [-1, 0, 1]:
            ly = self.left_paddle_y + dy
            ry = self.right_paddle_y + dy
            if 0 <= ly < self.height:
                grid[ly, 0] = 1
            if 0 <= ry < self.height:
                grid[ry, self.width - 1] = 1
        grid[self.ball_y, self.ball_x] = 2
        return grid

def plot_env(env, title='Mini Pong'):
    plt.imshow(env.render_array(), cmap='viridis', vmin=0, vmax=2)
    plt.xticks(range(env.width)); plt.yticks(range(env.height))
    plt.title(title)
    plt.show()

In [ ]:
env = MiniPongEnv(seed=RANDOM_STATE)
plot_env(env, 'Startzustand: links Auto-Paddle, rechts Agent')
print('State:', env.state())
print('Aktionen:', ACTION_NAMES)

## 3. Random Agent als Basislinie

Der Random Agent bewegt das rechte Paddle zufällig. Wenn Q-Learning funktioniert, sollte die gelernte Policy später deutlich mehr Treffer schaffen.

In [ ]:
def random_policy(state, rng):
    return int(rng.choice(ACTIONS))

def run_episode(env, policy, max_steps=250, render=False, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    state = env.reset()
    total_reward = 0.0
    frames = []
    final_info = {'agent_hits': 0}
    for _ in range(max_steps):
        action = policy(state, rng)
        state, reward, done, info = env.step(action)
        total_reward += reward
        final_info = info
        if render:
            frames.append(env.render_array().copy())
        if done:
            break
    return total_reward, final_info.get('agent_hits', 0), frames

random_results = [run_episode(MiniPongEnv(seed=RANDOM_STATE+i), random_policy, seed=RANDOM_STATE+i)[:2] for i in range(100)]
print('Random Agent: mittlerer Reward =', round(np.mean([r for r, h in random_results]), 3))
print('Random Agent: mittlere Treffer rechts =', round(np.mean([h for r, h in random_results]), 3))

## 4. Q-Learning

Die Q-Tabelle bewertet Aktionen für diskrete Zustände. Der Agent probiert anfangs zufällig aus (`epsilon`) und nutzt nach und nach häufiger die beste bekannte Aktion.

In [ ]:
def choose_action(q_table, state, epsilon, rng):
    if rng.random() < epsilon or state not in q_table:
        return int(rng.choice(ACTIONS))
    return ACTIONS[int(np.argmax(q_table[state]))]

def train_q_learning(alpha=0.35, gamma=0.95, epsilon=0.25, episodes=900, width=12, height=8, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    q_table = {}
    rewards, hits = [], []
    for episode in range(int(episodes)):
        env = MiniPongEnv(width=int(width), height=int(height), seed=seed + episode)
        state = env.reset()
        total_reward = 0.0
        done = False
        info = {'agent_hits': 0}
        while not done:
            q_table.setdefault(state, np.zeros(len(ACTIONS)))
            action = choose_action(q_table, state, epsilon, rng)
            next_state, reward, done, info = env.step(action)
            q_table.setdefault(next_state, np.zeros(len(ACTIONS)))
            a_idx = ACTIONS.index(action)
            target = reward + gamma * np.max(q_table[next_state]) * (not done)
            q_table[state][a_idx] += alpha * (target - q_table[state][a_idx])
            state = next_state
            total_reward += reward
        rewards.append(total_reward)
        hits.append(info.get('agent_hits', 0))
    return q_table, np.array(rewards), np.array(hits)

def q_policy_from_table(q_table):
    def policy(state, rng):
        if state not in q_table:
            return 0
        return ACTIONS[int(np.argmax(q_table[state]))]
    return policy

q_table, rewards, hits = train_q_learning()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(pd.Series(rewards).rolling(40, min_periods=1).mean())
axes[0].set_title('Reward, gleitender Mittelwert')
axes[0].set_xlabel('Episode')
axes[1].plot(pd.Series(hits).rolling(40, min_periods=1).mean())
axes[1].set_title('Treffer rechts, gleitender Mittelwert')
axes[1].set_xlabel('Episode')
plt.tight_layout(); plt.show()

In [ ]:
def evaluate(policy, episodes=100, width=12, height=8):
    rewards, hits = [], []
    for i in range(episodes):
        reward, hit_count, _ = run_episode(MiniPongEnv(width=width, height=height, seed=RANDOM_STATE + i), policy, seed=RANDOM_STATE + i)
        rewards.append(reward)
        hits.append(hit_count)
    return float(np.mean(rewards)), float(np.mean(hits))

before_reward, before_hits = evaluate(random_policy)
after_reward, after_hits = evaluate(q_policy_from_table(q_table))
print(f'Vor Training / Random: Reward={before_reward:.3f}, Treffer={before_hits:.2f}')
print(f'Nach Training / Q-Policy: Reward={after_reward:.3f}, Treffer={after_hits:.2f}')

## 5. Schlechte und bessere Trainingsläufe vergleichen

Hier sieht man deutlicher, ob der Agent etwas lernt: Wenige Episoden oder sehr viel Zufall führen meist zu wenig Treffern. Eine solide Baseline trifft den Ball häufiger.

In [ ]:
settings = {
    'zu wenig Training': dict(alpha=0.35, gamma=0.95, epsilon=0.25, episodes=50),
    'zu viel Zufall': dict(alpha=0.35, gamma=0.95, epsilon=0.85, episodes=900),
    'solide Baseline': dict(alpha=0.35, gamma=0.95, epsilon=0.15, episodes=1200),
}
rows = []
for name, params in settings.items():
    q_tmp, r_tmp, h_tmp = train_q_learning(**params)
    eval_reward, eval_hits = evaluate(q_policy_from_table(q_tmp), episodes=80)
    rows.append({'Einstellung': name, 'Training Treffer letzte 80': np.mean(h_tmp[-80:]), 'Evaluation Reward': eval_reward, 'Evaluation Treffer': eval_hits})
display(pd.DataFrame(rows))

## 6. Slider: Hyperparameter live verändern

Die Box trainiert neu. Achte besonders auf die Trefferkurve: Wenn sie steigt, lernt der Agent sichtbar, das rechte Paddle besser zum Ball zu bewegen.

In [ ]:
def q_learning_widget(alpha=0.35, gamma=0.95, epsilon=0.25, episodenanzahl=900, spielfeldhoehe=8):
    q, r, h = train_q_learning(alpha=alpha, gamma=gamma, epsilon=epsilon, episodes=episodenanzahl, height=spielfeldhoehe)
    eval_reward, eval_hits = evaluate(q_policy_from_table(q), episodes=80, height=spielfeldhoehe)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(pd.Series(r).rolling(40, min_periods=1).mean())
    axes[0].set_title(f'Reward | Evaluation {eval_reward:.2f}')
    axes[0].set_xlabel('Episode')
    axes[1].plot(pd.Series(h).rolling(40, min_periods=1).mean())
    axes[1].set_title(f'Treffer | Evaluation {eval_hits:.2f}')
    axes[1].set_xlabel('Episode')
    plt.tight_layout(); plt.show()

widgets.interact(
    q_learning_widget,
    alpha=widgets.FloatSlider(value=0.35, min=0.05, max=0.9, step=0.05),
    gamma=widgets.FloatSlider(value=0.95, min=0.1, max=0.99, step=0.05),
    epsilon=widgets.FloatSlider(value=0.25, min=0.0, max=0.9, step=0.05),
    episodenanzahl=widgets.IntSlider(value=900, min=100, max=2000, step=100),
    spielfeldhoehe=widgets.IntSlider(value=8, min=6, max=12),
);

## 7. Episode visualisieren

Die Visualisierung stoppt spätestens nach 100 Schritten. Zuerst gibt es einen eigenen Block für den Random Agent als Negativbeispiel. Danach folgt ein separater Block für den trainierten Q-Agenten.

In [ ]:
def visualize_episode(policy, titel='Agent', pause=0.10, max_visual_steps=100):
    env = MiniPongEnv(seed=RANDOM_STATE + 999)
    state = env.reset()
    total_reward = 0
    info = {'agent_hits': 0}
    for step in range(min(env.max_steps, max_visual_steps)):
        clear_output(wait=True)
        plot_env(env, title=f'{titel} | Schritt {step} | Treffer rechts: {info.get("agent_hits", 0)}')
        action = policy(state, np.random.default_rng(RANDOM_STATE + step))
        state, reward, done, info = env.step(action)
        total_reward += reward
        plt.pause(pause)
        if done:
            clear_output(wait=True)
            plot_env(env, title=f'{titel} Ende nach {step+1} Schritten | Treffer: {info.get("agent_hits", 0)} | Reward={total_reward:.2f}')
            break
    else:
        clear_output(wait=True)
        plot_env(env, title=f'{titel} Stopp nach {max_visual_steps} Schritten | Treffer: {info.get("agent_hits", 0)} | Reward={total_reward:.2f}')

### Random Agent visualisieren

Dieser Block zeigt absichtlich den untrainierten Random Agent. Er bewegt das Paddle zufällig und ist das Negativbeispiel: So soll sichtbar werden, dass ohne Lernen keine stabile Strategie entsteht.

In [ ]:
visualize_episode(random_policy, titel='Random Agent', pause=0.05, max_visual_steps=100)

### Trainierten Q-Agent visualisieren

Dieser Block nutzt die gelernte Q-Tabelle. Im Vergleich zum Random Agent sollte das rechte Paddle häufiger passend zum Ball stehen und mehr Treffer erzielen.

In [ ]:
visualize_episode(q_policy_from_table(q_table), titel='Trainierter Q-Agent', pause=0.05, max_visual_steps=100)

## 8. Reflexion

Warum ist echtes Pong immer noch schwerer?

- Echte Bilddaten bestehen aus Pixeln statt aus wenigen diskreten Zahlen.
- Der Agent muss Ball, Schläger und Geschwindigkeit erst aus Bildern erkennen.
- Das Timing ist kontinuierlicher und längerfristige Strategien werden wichtiger.
- Für echte Atari-Umgebungen nutzt man meistens neuronale Netze statt einer kleinen Q-Tabelle.

**Reflexionsfrage:** Welche zusätzlichen Informationen müsste ein Agent aus einem echten Videobild lernen, bevor er gute Aktionen wählen kann?